# Part 5 | Session 02: 바이브 코딩(Vibe Coding)이란?

**© Copyright AIDENTIFY. All rights reserved.**

## 1️⃣ 바이브 코딩 개요

**바이브 코딩(Vibe Coding)**은 Tesla AI 총괄이자 OpenAI 공동 창업자인 **Andrej Karpathy**가 2025년 2월에 제안한 개념입니다.

> "There's a new kind of coding I call 'vibe coding', where you fully give in to the vibes, embrace exponentials, and forget that the code even exists."
> — Andrej Karpathy

### 핵심 아이디어

- **자연어로 의도를 전달**하면 AI가 코드를 생성하는 새로운 프로그래밍 패러다임
- 개발자는 코드의 세부 구현보다 **"무엇을 만들 것인가"**에 집중
- AI를 **페어 프로그래밍 파트너**로 활용
- 전통적 코딩: 문법 암기 → 타이핑 → 디버깅
- 바이브 코딩: 의도 설명 → AI 생성 → 검증 → 반복

### 바이브 코딩이 주목받는 이유

| 관점 | 설명 |
|------|------|
| **생산성 향상** | 반복적인 보일러플레이트 코드 작성 시간 대폭 절감 |
| **진입 장벽 감소** | 비개발자도 프로토타이핑 가능 |
| **창의성 집중** | 구현 디테일보다 문제 해결과 설계에 집중 |
| **빠른 프로토타이핑** | 아이디어를 빠르게 동작하는 코드로 전환 |

## 2️⃣ 주요 AI 코딩 도구

현재 바이브 코딩을 지원하는 대표적인 AI 코딩 도구들을 비교해 보겠습니다.

| 도구 | 개발사 | 특징 | 형태 |
|------|--------|------|------|
| **GitHub Copilot** | GitHub (Microsoft) | VS Code 통합, 인라인 코드 제안, Chat 기능 | IDE 확장 |
| **Cursor** | Anysphere | AI-native IDE, 코드베이스 전체 이해, Composer 모드 | 독립 IDE |
| **Windsurf** | Codeium | Cascade 기능으로 멀티파일 편집, Flow 모드 | 독립 IDE |
| **Claude Code** | Anthropic | CLI 기반, 터미널에서 직접 사용, agentic 코딩 | CLI 도구 |
| **Replit Agent** | Replit | 웹 기반, 자연어로 전체 앱 생성, 배포까지 자동화 | 웹 IDE |

### 도구 선택 가이드

- **입문자 / 빠른 프로토타이핑** → Replit Agent
- **기존 VS Code 사용자** → GitHub Copilot 또는 Cursor
- **복잡한 프로젝트 리팩토링** → Cursor (Composer 모드)
- **터미널 중심 워크플로우** → Claude Code
- **멀티파일 대규모 수정** → Windsurf (Cascade)

## 3️⃣ 바이브 코딩 워크플로우

바이브 코딩의 일반적인 워크플로우는 다음과 같습니다:

```
1. 자연어로 요구사항 작성
   ↓
2. AI가 코드 생성
   ↓
3. 생성된 코드 검증 (실행, 테스트)
   ↓
4. 피드백 & 수정 요청
   ↓
5. 반복 (원하는 결과 나올 때까지)
```

### 핵심 원칙
- **작은 단위로 반복**: 한 번에 너무 큰 작업을 요청하지 않기
- **결과를 항상 검증**: AI가 생성한 코드를 그대로 신뢰하지 않기
- **컨텍스트 유지**: 대화의 흐름을 유지하며 점진적으로 발전시키기

### 실습 준비: "바이브 코딩 파트너" 최소 구현

이 노트북에서는 **OpenAI `gpt-4o-mini`** 를 바이브 코딩 파트너로 삼아, Claude Code·Cursor 같은 도구가 내부에서 하는 일을
가장 작은 형태로 직접 만들어 봅니다.

```
generate_code(요구사항)  →  Python 코드 문자열
run_code(코드)          →  (네임스페이스, 표준출력, 에러)
```

두 함수만 있으면 위 워크플로우의 1~5단계를 코드로 돌려볼 수 있습니다.
`.env` 에 `OPENAI_API_KEY` 가 없으면 미리 준비된 템플릿 코드로 대신 실행됩니다.

In [ ]:
# 실습 준비 — 같은 폴더의 .env 에서 OPENAI_API_KEY 로드 (키 값 자체는 출력하지 않음)
import os
from dotenv import load_dotenv

load_dotenv()

MODEL = "gpt-4o-mini"
HAS_OPENAI = os.environ.get("OPENAI_API_KEY", "").startswith("sk-")

if HAS_OPENAI:
    from openai import OpenAI
    client = OpenAI()
    print(f"✅ OpenAI 연결 준비 완료 — 모델: {MODEL}")
else:
    client = None
    print("ℹ️ OPENAI_API_KEY 없음 — 아래 셀들은 미리 준비된 템플릿 코드로 대신 실행됩니다.")

In [ ]:
# 바이브 코딩 파트너의 최소 구현: (1) 자연어 → 코드 생성, (2) 코드 실행 & 결과 캡처
import io, re, contextlib, traceback

SYSTEM_PROMPT = """당신은 숙련된 Python 개발자입니다.
사용자의 요구사항을 만족하는 Python 코드만 출력하세요.
- 설명 없이 ```python 코드블록 하나만 출력합니다.
- 표준 라이브러리만 사용합니다.
- 함수/클래스에는 한 줄 docstring 을 답니다."""


def generate_code(prompt: str, history: list | None = None) -> str:
    """자연어 요구사항(prompt)을 LLM 에 보내 Python 코드 문자열을 돌려받는다.
    history 를 넘기면 이전 대화(요구사항·생성 코드)를 이어서 요청한다."""
    if not HAS_OPENAI:
        return _template_code(prompt)
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages += history or []
    messages.append({"role": "user", "content": prompt})
    resp = client.chat.completions.create(model=MODEL, messages=messages, temperature=0)
    text = resp.choices[0].message.content
    m = re.search(r"```(?:python)?\n(.*?)```", text, re.S)   # 코드블록 안쪽만 추출
    return (m.group(1) if m else text).strip()


def run_code(code: str, namespace: dict | None = None) -> tuple[dict, str, str | None]:
    """코드를 실행하고 (네임스페이스, 표준출력, 에러 traceback 또는 None) 을 돌려준다."""
    ns = namespace if namespace is not None else {}
    ns.setdefault("__name__", "vibe_sandbox")
    buf, error = io.StringIO(), None
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, ns)
    except Exception:
        error = traceback.format_exc()
    return ns, buf.getvalue(), error


# ---- API 키가 없을 때 쓰는 템플릿 (프롬프트에 포함된 키워드로 선택) ----
_TEMPLATES = [
    ("이메일", '''import re

def extract_emails(text: str) -> list[str]:
    """텍스트에서 이메일 주소를 추출해 중복 제거된 리스트로 반환한다."""
    pattern = r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}"
    return list(set(re.findall(pattern, text)))'''),
    ("TaskManager 를 검증", '''m = TaskManager()
m.add("A", priority=2); m.add("B", priority=1); m.complete(0)
assert [t.title for t in m.list_pending()] == ["B"]
print("모든 테스트 통과")'''),
    ("TaskManager", '''class TaskManager:
    """Task 목록을 관리하는 클래스"""
    def __init__(self):
        self.tasks: list[Task] = []
    def add(self, title: str, description: str = "", priority: int = 3) -> Task:
        task = Task(title=title, description=description, priority=priority)
        self.tasks.append(task)
        return task
    def complete(self, index: int):
        self.tasks[index].completed = True
    def list_pending(self) -> list[Task]:
        return sorted([t for t in self.tasks if not t.completed], key=lambda t: t.priority)
    def summary(self):
        done = sum(t.completed for t in self.tasks)
        print(f"전체 {len(self.tasks)}개 / 완료 {done}개 / 미완료 {len(self.tasks) - done}개")'''),
    ("Task", '''from dataclasses import dataclass, field
from datetime import datetime

@dataclass
class Task:
    """할 일 관리를 위한 Task 데이터 클래스"""
    title: str
    description: str = ""
    priority: int = 3
    completed: bool = False
    created_at: datetime = field(default_factory=datetime.now)
    def __str__(self):
        status = "[완료]" if self.completed else "[미완료]"
        return f"{status} [우선순위:{self.priority}] {self.title}"'''),
    ("slugify", '''import re

def slugify(text: str) -> str:
    """문자열을 URL 슬러그로 변환한다 (유니코드 문자 유지)."""
    text = re.sub(r"[^\\w\\s-]", "", text.lower())
    return re.sub(r"[\\s_-]+", "-", text).strip("-")'''),
    ("clean_csv", '''import csv, io
from datetime import date

def clean_csv(text: str) -> list[dict]:
    """CSV 문자열을 파싱해 score 결측치를 평균으로 채우고 date 를 date 객체로 바꾼다."""
    rows = list(csv.DictReader(io.StringIO(text)))
    scores = [float(r["score"]) for r in rows if r["score"] != ""]
    mean = sum(scores) / len(scores) if scores else 0.0
    for r in rows:
        r["score"] = float(r["score"]) if r["score"] != "" else mean
        r["date"] = date.fromisoformat(r["date"])
    return rows'''),
    ("데이터 처리", '''def process_data(data):
    """데이터를 처리한다 (요구사항이 모호해 AI 가 임의로 해석한 예)"""
    return [x for x in data if x is not None]'''),
]


def _template_code(prompt: str) -> str:
    for keyword, snippet in _TEMPLATES:
        if keyword in prompt:
            return snippet
    return 'print("(템플릿 없음 — OPENAI_API_KEY 를 설정하면 LLM 이 코드를 생성합니다)")'

print("generate_code / run_code 준비 완료")

### 데모 1: 자연어 → 코드 → 실행 → 검증

워크플로우 1~3단계를 그대로 돌려봅니다. AI 가 만든 코드를 **눈으로 읽는 데서 그치지 않고 실제로 실행해 검증**하는 것이 핵심입니다.

**프롬프트 예시:**
> "주어진 텍스트에서 이메일 주소를 모두 추출하는 Python 함수를 만들어줘. 정규표현식을 사용하고, 결과는 중복 제거된 리스트로 반환해줘."


In [ ]:
# 데모 1: 자연어 요구사항 → 코드 생성 → 실행 → 검증
prompt = ("주어진 텍스트에서 이메일 주소를 모두 추출하는 Python 함수 extract_emails(text) 를 만들어줘. "
          "정규표현식을 사용하고, 결과는 중복 제거된 리스트로 반환해줘.")

generated = generate_code(prompt)
print("=== AI 가 생성한 코드 ===")
print(generated)

ns, out, err = run_code(generated)
if err:
    print(err)
else:
    sample_text = """
    문의사항은 support@example.com으로 보내주세요.
    담당자: kim@aidentify.io, lee@aidentify.io
    기존 문의: support@example.com
    """
    result = ns["extract_emails"](sample_text)
    print("\n=== 검증 ===")
    print("추출된 이메일:", sorted(result))
    assert set(result) == {"support@example.com", "kim@aidentify.io", "lee@aidentify.io"}
    print(f"✅ 총 {len(result)}개의 고유 이메일 — 요구사항 충족")

## 4️⃣ 효과적인 프롬프팅 기법

바이브 코딩에서 가장 중요한 것은 **AI에게 얼마나 명확하게 의도를 전달하느냐**입니다.

### 기법 1: 구체적 지시

| 나쁜 예 | 좋은 예 |
|---------|--------|
| "데이터 처리 함수 만들어줘" | "CSV 파일을 읽어서 결측치를 평균으로 대체하고, 날짜 컬럼을 datetime으로 변환하는 함수를 만들어줘" |
| "API 만들어줘" | "FastAPI로 사용자 CRUD API를 만들어줘. Pydantic 모델을 사용하고, 에러 핸들링도 포함해줘" |

### 기법 2: 컨텍스트 제공

- 사용 중인 **프레임워크/라이브러리** 명시
- **입력/출력 형태** 구체적으로 설명
- **제약 조건** 언급 (성능, 메모리, 호환성 등)

### 기법 3: 단계적 접근

1. 먼저 **전체 구조/설계**를 요청
2. 각 모듈을 **하나씩** 구현 요청
3. **테스트 코드** 작성 요청
4. **리팩토링/최적화** 요청

### 실습 A: 나쁜 프롬프트 vs 좋은 프롬프트

같은 의도를 두 가지 프롬프트로 요청하고 생성 결과를 비교합니다.
나쁜 프롬프트는 AI 가 빈칸을 **임의로 채우므로** 결과를 검증할 기준조차 없고, 좋은 프롬프트는 **함수명·입출력·제약**이 고정되어 있어 바로 테스트할 수 있습니다.

In [ ]:
# 같은 의도, 다른 프롬프트 — 생성 결과가 얼마나 달라지는지 비교
bad_prompt = "데이터 처리 함수 만들어줘"
good_prompt = ("CSV 형식 문자열을 받아 csv 모듈로 파싱하는 함수 clean_csv(text) 를 만들어줘. "
               "'score' 컬럼은 float 로 변환하되 빈 값은 나머지 score 들의 평균으로 채우고, "
               "'date' 컬럼은 datetime.date 객체로 변환해서 dict 의 리스트로 반환해. pandas 는 쓰지 마.")

for label, p in [("나쁜 예", bad_prompt), ("좋은 예", good_prompt)]:
    print(f"{'=' * 25} {label}: \"{p[:45]}...\" {'=' * 25}")
    print(generate_code(p), "\n")

In [ ]:
# 좋은 프롬프트의 결과는 입출력이 명확하므로 곧바로 실행해 검증할 수 있다
csv_text = """name,score,date
kim,90,2026-09-19
lee,,2026-09-19
park,70,2026-09-20
"""
ns, out, err = run_code(generate_code(good_prompt))
if err:
    print("❌ 실행 실패 — 다음 실습의 '피드백 루프' 가 필요한 이유입니다\n", err)
else:
    for row in ns["clean_csv"](csv_text):
        print(row)

### 실습 B: 단계적 접근 — 대화 컨텍스트를 유지하며 설계 → 구현 → 테스트

한 번에 "할 일 관리 앱 만들어줘" 라고 하지 않고, **이전 요청과 생성 코드를 대화 history 에 누적**하면서 세 단계로 나눠 요청합니다.
Claude Code 가 세션 안에서 이전 작업을 기억하는 것과 같은 원리입니다.

In [ ]:
# 단계적 접근: history(대화 기록)와 sandbox(누적 실행 네임스페이스)를 유지하며 요청
history: list[dict] = []
sandbox: dict = {}


def step(prompt: str) -> None:
    generated = generate_code(prompt, history)
    history.extend([{"role": "user", "content": prompt},
                    {"role": "assistant", "content": f"```python\n{generated}\n```"}])
    _, out, err = run_code(generated, sandbox)
    print(f"▶ 요청: {prompt}\n\n{generated}\n\n[실행 결과]\n{out}{err or ''}" + "-" * 70)


step("할 일 관리를 위한 Task 데이터클래스를 만들어줘. 필드: title(str), description(str, 기본 ''), "
     "priority(int, 1=높음~5=낮음, 기본 3), completed(bool, 기본 False), created_at(datetime, 생성 시각 자동). "
     "__str__ 은 '[완료]/[미완료] [우선순위:n] 제목' 형식으로.")

step("위 Task 를 관리하는 TaskManager 클래스를 만들어줘. add(title, description='', priority=3) -> Task, "
     "complete(index), list_pending() -> 우선순위 오름차순으로 정렬된 미완료 Task 리스트, "
     "전체/완료/미완료 개수를 출력하는 summary(). Task 는 이미 정의되어 있으니 다시 정의하지 마.")

step("위 TaskManager 를 검증하는 테스트 코드를 assert 문으로 작성해줘. Task 3개 추가, 1개 완료, "
     "list_pending 정렬 확인, summary 호출까지 포함하고 마지막에 '모든 테스트 통과' 를 출력해. "
     "Task 와 TaskManager 는 이미 정의되어 있으니 다시 정의하지 마.")

### 실습 C: 검증 → 피드백 → 반복 루프 (미니 하네스)

워크플로우 3~5단계를 자동화합니다. 요구사항과 **테스트 케이스(입력, 기대값)** 를 주면, 생성된 코드를 실행해 검사하고
실패한 케이스의 실제값·기대값을 AI 에게 되돌려 고치게 합니다. 테스트가 통과하거나 최대 시도 횟수에 도달할 때까지 반복합니다.

이 루프가 바로 Claude Code·Cursor 같은 **에이전트형 코딩 도구가 내부에서 돌리는 하네스(harness)의 핵심**이며,
이 과정의 주제인 "Agentic AI 와 Harness 설계" 로 이어집니다.

In [ ]:
# 검증 → 피드백 → 반복: 테스트가 통과할 때까지 실패 케이스를 AI 에게 되돌려주는 미니 하네스
def check(ns: dict, func_name: str, cases: list[tuple]) -> list[str]:
    """cases 의 (입력, 기대값) 을 하나씩 검사해 실패 메시지 목록을 돌려준다 (비어 있으면 통과)."""
    fn = ns.get(func_name)
    if fn is None:
        return [f"함수 {func_name} 가 정의되지 않았습니다"]
    fails = []
    for arg, expected in cases:
        try:
            got = fn(arg)
        except Exception as e:
            got = f"{type(e).__name__}: {e}"
        if got != expected:
            fails.append(f"{func_name}({arg!r}) → {got!r}   (기대값: {expected!r})")
    return fails


def vibe_loop(spec: str, func_name: str, cases: list[tuple], max_iter: int = 3) -> str | None:
    """spec 으로 코드를 생성하고 cases 로 검증한다. 실패하면 실패 케이스를 피드백해 재생성한다."""
    history: list[dict] = []
    prompt = spec
    for i in range(1, max_iter + 1):
        generated = generate_code(prompt, history)
        history.extend([{"role": "user", "content": prompt},
                        {"role": "assistant", "content": f"```python\n{generated}\n```"}])
        ns, _, err = run_code(generated)
        fails = [err.strip().splitlines()[-1]] if err else check(ns, func_name, cases)
        print(f"── 시도 {i} ──\n{generated}\n")
        if not fails:
            print(f"✅ 테스트 {len(cases)}개 모두 통과 (시도 {i}회)")
            return generated
        print("❌ 실패 — AI 에게 피드백합니다:\n  " + "\n  ".join(fails) + "\n")
        prompt = ("위 코드를 테스트했더니 아래 케이스가 실패했어. 수정한 전체 코드를 다시 줘.\n"
                  + "\n".join(fails))
    print("⚠️ 최대 시도 횟수 초과 — 사람이 개입해야 합니다")
    return None


spec = "문자열을 URL 슬러그로 바꾸는 함수 slugify(text) 를 만들어줘."
cases = [
    ("Hello World", "hello-world"),
    ("  Hello   World  ", "hello-world"),
    ("Hello, World!", "hello-world"),
    ("Python 3.11 출시", "python-311-출시"),   # 한글은 유지, 구두점은 제거
    ("a -- b", "a-b"),
    ("", ""),
]
final_code = vibe_loop(spec, "slugify", cases)

**관찰 포인트**

- 요구사항(spec)에는 "한글 유지" 같은 세부 조건이 없습니다. 이런 빈틈은 **테스트가 대신 말해 주고**, AI 는 에러를 보고 고칩니다.
- 첫 시도에 통과했다면 spec 을 더 모호하게 바꾸거나 테스트를 더 까다롭게 만들어 다시 실행해 보세요 (예: `"C++ & Go"` → `"c-go"`).
- `max_iter` 를 넘기면 루프가 멈추고 사람에게 넘깁니다. **끝없이 AI 에게 맡기지 않는 것**도 하네스 설계의 일부입니다.

## 5️⃣ 바이브 코딩의 장단점 및 주의사항

### 장점

- **개발 속도 향상**: 보일러플레이트 코드 자동 생성으로 핵심 로직에 집중
- **학습 도구**: 새로운 언어나 프레임워크를 빠르게 학습 가능
- **프로토타이핑**: 아이디어를 빠르게 검증할 수 있음
- **코드 품질**: AI가 베스트 프랙티스를 자연스럽게 반영

### 단점 및 주의사항

- **환각(Hallucination)**: AI가 존재하지 않는 API나 라이브러리를 사용할 수 있음
- **보안 취약점**: 생성된 코드에 보안 이슈가 포함될 수 있음
- **과도한 의존**: 기본기 없이 AI에만 의존하면 디버깅이 어려워짐
- **라이선스 문제**: AI가 학습한 코드의 라이선스 이슈 가능성
- **컨텍스트 한계**: 대규모 프로젝트에서는 AI가 전체 맥락을 파악하기 어려움

### 바이브 코딩을 잘 활용하기 위한 팁

1. **기본기를 갖추세요**: AI가 생성한 코드를 이해하고 검증할 수 있어야 합니다
2. **항상 테스트하세요**: AI 생성 코드는 반드시 실행하고 테스트해야 합니다
3. **보안을 확인하세요**: 민감한 정보(API 키, 비밀번호)가 포함되지 않았는지 확인
4. **버전 관리를 하세요**: Git을 활용하여 변경 이력을 추적하세요
5. **점진적으로 진행하세요**: 작은 단위로 요청하고 검증하는 반복 사이클을 유지하세요

> **다음 세션(Session 03)**에서는 Claude Code를 활용한 실제 AI Agent 구현 실습을 진행합니다.

---

**© Copyright AIDENTIFY. All rights reserved.**